# Duplicate Media Audit

Finds duplicate uploads by stripping datetime prefixes from filenames to reveal the **base name** (e.g. `2025_07_27_13_03_45_IMG_8345.MOV` → `IMG_8345.MOV`).

For each duplicate group, determines:
- **Which to keep** — the one with the most metadata (tags, watch progress, better `recorded_at`)
- **What to transfer** — tags, watch progress, and `recorded_at` time from the duplicate to the keeper
- **What to delete** — the duplicate DB record + its R2 files

### Steps

1. **Connect** to Supabase + R2
2. **Find duplicates** by base filename + file size
3. **Analyze each group** — compare metadata richness to pick the keeper
4. **Export** cleanup plan to CSV
5. **Execute** cleanup (transfer metadata, delete duplicates)

In [1]:
import os
import re
from collections import defaultdict
from dotenv import load_dotenv
from tabulate import tabulate

load_dotenv()

def clean_env(key):
    val = os.getenv(key, "")
    return val.split("#")[0].strip() if val else ""

SUPABASE_URL = clean_env("VITE_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = clean_env("VITE_SUPABASE_SERVICE_ROLE_KEY")
R2_ACCOUNT_ID = clean_env("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = clean_env("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = clean_env("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = clean_env("R2_BUCKET_NAME")

missing = [k for k, v in {
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}.items() if not v]

if missing:
    print(f"Missing config: {', '.join(missing)}")
else:
    print("All config loaded.")

All config loaded.


In [2]:
import boto3
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase connected.")

r2 = boto3.client(
    service_name="s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
print("R2 connected.")

Supabase connected.
R2 connected.


---
## Step 1: Fetch All Data

In [3]:
# ── Fetch all media records ──
all_media = []
offset = 0
while True:
    res = sb.table("media").select(
        "id, title, description, storage_path, thumbnail_path, original_filename, "
        "file_size_bytes, mime_type, recorded_at, created_at, duration, resolution"
    ).range(offset, offset + 1000 - 1).execute()
    all_media.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

# ── Fetch all media_tags ──
all_tags = []
offset = 0
while True:
    res = sb.table("media_tags").select(
        "id, media_id, tag_id, start_time, end_time"
    ).range(offset, offset + 1000 - 1).execute()
    all_tags.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

# ── Fetch all watch_progress ──
all_progress = []
offset = 0
while True:
    res = sb.table("watch_progress").select(
        "id, user_id, media_id, position, updated_at"
    ).range(offset, offset + 1000 - 1).execute()
    all_progress.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

# ── Build lookups ──
tags_by_media = defaultdict(list)
for t in all_tags:
    tags_by_media[t["media_id"]].append(t)

progress_by_media = defaultdict(list)
for p in all_progress:
    progress_by_media[p["media_id"]].append(p)

media_by_id = {r["id"]: r for r in all_media}

print(f"Loaded {len(all_media)} media, {len(all_tags)} tag assignments, {len(all_progress)} watch progress entries.")

Loaded 478 media, 629 tag assignments, 7 watch progress entries.


---
## Step 2: Find Duplicates by Base Filename

Strips datetime prefixes (`YYYY_MM_DD_HH_MM_SS_`) from filenames to find the base name, then groups by base name + file size.

In [4]:
# Pattern: YYYY_MM_DD_HH_MM_SS_ prefix added by bulk upload
DATETIME_PREFIX_RE = re.compile(r'^\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_')

def get_base_filename(filename: str) -> str:
    """Strip datetime prefix to get the original iPhone filename."""
    if not filename:
        return filename
    return DATETIME_PREFIX_RE.sub('', filename)

def format_bytes(size):
    if not size:
        return "—"
    for unit in ["B", "KB", "MB", "GB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"

def is_midnight(recorded_at_str):
    if not recorded_at_str:
        return True
    return "00:00:00" in recorded_at_str

def extract_datetime_from_filename(filename):
    """Extract YYYY_MM_DD_HH_MM_SS from filename prefix."""
    m = re.match(r'(\d{4})_(\d{2})_(\d{2})_(\d{2})_(\d{2})_(\d{2})_', filename or '')
    if m:
        y, mo, d, h, mi, s = m.groups()
        return f"{y}-{mo}-{d}T{h}:{mi}:{s}"
    return None

# ── Group by (base_filename, file_size) ──
groups = defaultdict(list)
for r in all_media:
    fn = r.get("original_filename", "")
    base = get_base_filename(fn)
    size = r.get("file_size_bytes")
    if base and size:
        groups[(base, size)].append(r)

# Filter to groups with >1 record (actual duplicates)
dupe_groups = {k: v for k, v in groups.items() if len(v) > 1}

# Also check for same base filename regardless of size (catches re-encoded copies)
name_only_groups = defaultdict(list)
for r in all_media:
    fn = r.get("original_filename", "")
    base = get_base_filename(fn)
    if base:
        name_only_groups[base].append(r)

name_only_dupes = {k: v for k, v in name_only_groups.items() if len(v) > 1}
# Find groups that are name-only but NOT in the size-confirmed set
size_confirmed_ids = set()
for recs in dupe_groups.values():
    for r in recs:
        size_confirmed_ids.add(r["id"])

name_only_suspicious = {}
for base, recs in name_only_dupes.items():
    # Only include if NOT all already in size-confirmed groups
    if not all(r["id"] in size_confirmed_ids for r in recs):
        name_only_suspicious[base] = recs

# ── Summary ──
total_dupe_records = sum(len(v) for v in dupe_groups.values())
total_extra = total_dupe_records - len(dupe_groups)  # records that can be removed

print("=" * 60)
print("DUPLICATE DETECTION")
print("=" * 60)
print()
print(f"  Total media records:         {len(all_media)}")
print(f"  Duplicate groups (confirmed): {len(dupe_groups)}")
print(f"  Records in duplicate groups:  {total_dupe_records}")
print(f"  Extra copies to remove:       {total_extra}")
if name_only_suspicious:
    print(f"  Suspicious (name match only): {len(name_only_suspicious)} groups")

DUPLICATE DETECTION

  Total media records:         478
  Duplicate groups (confirmed): 119
  Records in duplicate groups:  238
  Extra copies to remove:       119
  Suspicious (name match only): 2 groups


---
## Step 3: Analyze Each Duplicate Group

For each group, picks the **keeper** based on:
1. Most tags assigned
2. Has watch progress
3. Has better `recorded_at` (full datetime vs midnight)
4. Has more metadata (description, duration, resolution)
5. Earliest `created_at` (tiebreaker — original upload)

Then identifies what metadata the duplicate has that the keeper is missing.

In [5]:
def score_record(r):
    """Score a record by metadata richness. Higher = more worth keeping."""
    rid = r["id"]
    score = 0
    
    # Tags are the most important — user effort went into assigning them
    score += len(tags_by_media[rid]) * 10
    
    # Watch progress means the user interacted with this copy
    score += len(progress_by_media[rid]) * 5
    
    # Full datetime in recorded_at (not midnight)
    if r.get("recorded_at") and not is_midnight(r["recorded_at"]):
        score += 3
    
    # Has description
    if r.get("description"):
        score += 2
    
    # Has duration/resolution (processed metadata)
    if r.get("duration"):
        score += 1
    if r.get("resolution"):
        score += 1
    
    return score


# ── Build cleanup plan ──
cleanup_plan = []  # list of dicts with keeper, dupes, transfers

for (base_name, file_size), records in sorted(dupe_groups.items()):
    # Score each record
    scored = [(score_record(r), r) for r in records]
    # Sort: highest score first, then earliest created_at as tiebreaker
    scored.sort(key=lambda x: (-x[0], x[1]["created_at"]))
    
    keeper = scored[0][1]
    keeper_score = scored[0][0]
    dupes = [r for _, r in scored[1:]]
    
    # ── Determine what to transfer from dupes to keeper ──
    transfers = []
    
    # 1. recorded_at: if keeper is midnight but a dupe has full datetime
    keeper_midnight = is_midnight(keeper.get("recorded_at"))
    best_recorded_at = None
    best_recorded_at_source = None
    
    if keeper_midnight:
        for d in dupes:
            # Check if dupe has full datetime in recorded_at
            if d.get("recorded_at") and not is_midnight(d["recorded_at"]):
                best_recorded_at = d["recorded_at"]
                best_recorded_at_source = d["id"]
                break
        
        # Also check if any dupe's filename has embedded datetime
        if not best_recorded_at:
            for d in dupes:
                dt = extract_datetime_from_filename(d.get("original_filename", ""))
                if dt:
                    best_recorded_at = dt + "+00:00"
                    best_recorded_at_source = d["id"] + " (filename)"
                    break
        
        # Check keeper's own filename too
        if not best_recorded_at:
            dt = extract_datetime_from_filename(keeper.get("original_filename", ""))
            if dt:
                best_recorded_at = dt + "+00:00"
                best_recorded_at_source = "keeper filename"
        
        if best_recorded_at:
            transfers.append({
                "type": "recorded_at",
                "value": best_recorded_at,
                "source": best_recorded_at_source,
            })
    
    # 2. Tags: collect tags from dupes that keeper doesn't have
    keeper_tag_ids = {t["tag_id"] for t in tags_by_media[keeper["id"]]}
    tags_to_transfer = []
    for d in dupes:
        for t in tags_by_media[d["id"]]:
            if t["tag_id"] not in keeper_tag_ids:
                tags_to_transfer.append(t)
                keeper_tag_ids.add(t["tag_id"])  # prevent dupes
    
    if tags_to_transfer:
        transfers.append({
            "type": "tags",
            "count": len(tags_to_transfer),
            "details": tags_to_transfer,
        })
    
    # 3. Watch progress: transfer if keeper has none but dupe does
    keeper_progress = progress_by_media[keeper["id"]]
    progress_to_transfer = []
    keeper_progress_users = {p["user_id"] for p in keeper_progress}
    for d in dupes:
        for p in progress_by_media[d["id"]]:
            if p["user_id"] not in keeper_progress_users:
                progress_to_transfer.append(p)
                keeper_progress_users.add(p["user_id"])
    
    if progress_to_transfer:
        transfers.append({
            "type": "watch_progress",
            "count": len(progress_to_transfer),
            "details": progress_to_transfer,
        })
    
    # 4. Description: if keeper has none but dupe does
    if not keeper.get("description"):
        for d in dupes:
            if d.get("description"):
                transfers.append({
                    "type": "description",
                    "value": d["description"],
                    "source": d["id"],
                })
                break
    
    # 5. Duration/resolution: if keeper is missing but dupe has it
    if not keeper.get("duration"):
        for d in dupes:
            if d.get("duration"):
                transfers.append({"type": "duration", "value": d["duration"], "source": d["id"]})
                break
    
    if not keeper.get("resolution"):
        for d in dupes:
            if d.get("resolution"):
                transfers.append({"type": "resolution", "value": d["resolution"], "source": d["id"]})
                break
    
    cleanup_plan.append({
        "base_name": base_name,
        "file_size": file_size,
        "keeper": keeper,
        "keeper_score": keeper_score,
        "dupes": dupes,
        "transfers": transfers,
    })

print(f"Built cleanup plan for {len(cleanup_plan)} duplicate groups.")

# ── Summary stats ──
total_transfers = sum(len(p["transfers"]) for p in cleanup_plan)
groups_with_transfers = sum(1 for p in cleanup_plan if p["transfers"])
recorded_at_transfers = sum(1 for p in cleanup_plan for t in p["transfers"] if t["type"] == "recorded_at")
tag_transfers = sum(t["count"] for p in cleanup_plan for t in p["transfers"] if t["type"] == "tags")
progress_transfers = sum(t["count"] for p in cleanup_plan for t in p["transfers"] if t["type"] == "watch_progress")

print(f"\n  Groups with metadata to transfer: {groups_with_transfers}")
print(f"  recorded_at to fill in:           {recorded_at_transfers}")
print(f"  Tag assignments to move:           {tag_transfers}")
print(f"  Watch progress to move:            {progress_transfers}")

Built cleanup plan for 119 duplicate groups.

  Groups with metadata to transfer: 1
  recorded_at to fill in:           1
  Tag assignments to move:           0
  Watch progress to move:            0


In [6]:
# ── Detailed view of each duplicate group ──

for i, plan in enumerate(cleanup_plan):
    keeper = plan["keeper"]
    print(f"\n{'━' * 80}")
    print(f"GROUP {i+1}: {plan['base_name']}  ({format_bytes(plan['file_size'])})")
    print(f"{'━' * 80}")
    
    # Show all records in this group
    rows = []
    all_in_group = [(keeper, "KEEP")] + [(d, "DELETE") for d in plan["dupes"]]
    for r, action in all_in_group:
        tags_count = len(tags_by_media[r["id"]])
        progress_count = len(progress_by_media[r["id"]])
        has_time = "yes" if (r.get("recorded_at") and not is_midnight(r["recorded_at"])) else "no"
        rows.append([
            action,
            r["original_filename"][:40],
            r.get("recorded_at", "—")[:19] if r.get("recorded_at") else "—",
            has_time,
            tags_count,
            progress_count,
            r["created_at"][:19],
        ])
    print(tabulate(rows, headers=["", "Filename", "recorded_at", "Has Time", "Tags", "Progress", "Uploaded"], tablefmt="simple"))
    
    # Show transfers
    if plan["transfers"]:
        print(f"\n  Transfers to keeper:")
        for t in plan["transfers"]:
            if t["type"] == "recorded_at":
                print(f"    → recorded_at: {t['value'][:19]}  (from {t['source']})")
            elif t["type"] == "tags":
                print(f"    → {t['count']} tag assignment(s)")
            elif t["type"] == "watch_progress":
                print(f"    → {t['count']} watch progress entry/entries")
            elif t["type"] == "description":
                print(f"    → description: {t['value'][:50]}")
            elif t["type"] in ("duration", "resolution"):
                print(f"    → {t['type']}: {t['value']}")
    else:
        print(f"\n  No metadata to transfer.")

print(f"\n\n{'=' * 80}")
print(f"TOTAL: {len(cleanup_plan)} groups, {sum(len(p['dupes']) for p in cleanup_plan)} records to delete")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GROUP 1: IMG_0198.MP4  (15.0 MB)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        Filename                          recorded_at          Has Time      Tags    Progress  Uploaded
------  --------------------------------  -------------------  ----------  ------  ----------  -------------------
KEEP    2025_11_28_12_19_23_IMG_0198.MP4  2025-11-28T12:19:23  yes              1           0  2026-03-22T21:19:31
DELETE  IMG_0198.MP4                      2025-11-28T20:19:23  yes              1           0  2026-03-22T21:25:09

  No metadata to transfer.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GROUP 2: IMG_0199.MP4  (12.6 MB)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        Filename                          recorded_at          Has Time      Tags    Progress  Uploaded
------  -------------------

---
## Step 3b: Suspicious Name-Only Matches

These share the same base filename but have **different file sizes** — could be re-encoded versions or coincidental name matches. Review manually.

In [7]:
if name_only_suspicious:
    print(f"SUSPICIOUS NAME-ONLY MATCHES ({len(name_only_suspicious)} groups)")
    print(f"{'─' * 80}")
    print("These have the same base filename but DIFFERENT file sizes.")
    print()

    for base, recs in sorted(name_only_suspicious.items()):
        print(f"\n  {base}:")
        rows = []
        for r in sorted(recs, key=lambda x: x["created_at"]):
            rows.append([
                r["original_filename"][:40],
                format_bytes(r.get("file_size_bytes")),
                r.get("resolution", "—"),
                r.get("recorded_at", "—")[:19] if r.get("recorded_at") else "—",
                r["created_at"][:19],
            ])
        print(tabulate(rows, headers=["Filename", "Size", "Resolution", "recorded_at", "Uploaded"], tablefmt="simple"))
else:
    print("No suspicious name-only matches found.")

SUSPICIOUS NAME-ONLY MATCHES (2 groups)
────────────────────────────────────────────────────────────────────────────────
These have the same base filename but DIFFERENT file sizes.


  IMG_1430.MP4:
Filename                          Size     Resolution    recorded_at          Uploaded
--------------------------------  -------  ------------  -------------------  -------------------
2024_08_02_17_16_16_IMG_1430.MP4  50.0 MB                2024-08-02T17:16:16  2026-03-22T10:21:43
IMG_1430.MP4                      77.7 MB  1180x2556     2026-02-23T02:06:57  2026-03-22T10:29:03

  IMG_8259.MOV:
Filename                          Size      Resolution    recorded_at          Uploaded
--------------------------------  --------  ------------  -------------------  -------------------
2025_07_25_15_04_07_IMG_8259.MOV  135.8 MB                2025-07-25T15:04:07  2026-03-22T20:53:29
IMG_8259.MOV                      87.1 MB                 2025-07-25T22:04:07  2026-03-22T22:08:54


---
## Step 4: Export Cleanup Plan to CSV

In [8]:
import csv
from pathlib import Path

output_path = Path(r"C:\Users\minds\Videos\DANCE\dance-library\TESTING\dedup_plan.csv")

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "group", "base_name", "action", "id", "original_filename",
        "file_size", "recorded_at", "has_time", "tags", "watch_progress",
        "created_at", "transfers"
    ])
    
    for i, plan in enumerate(cleanup_plan):
        keeper = plan["keeper"]
        transfer_summary = "; ".join(
            f"{t['type']}={t.get('value', t.get('count', ''))}" 
            for t in plan["transfers"]
        ) or "none"
        
        # Keeper row
        writer.writerow([
            i + 1, plan["base_name"], "KEEP", keeper["id"],
            keeper["original_filename"],
            keeper.get("file_size_bytes", ""),
            keeper.get("recorded_at", ""),
            "yes" if (keeper.get("recorded_at") and not is_midnight(keeper["recorded_at"])) else "no",
            len(tags_by_media[keeper["id"]]),
            len(progress_by_media[keeper["id"]]),
            keeper["created_at"],
            transfer_summary,
        ])
        
        # Dupe rows
        for d in plan["dupes"]:
            writer.writerow([
                i + 1, plan["base_name"], "DELETE", d["id"],
                d["original_filename"],
                d.get("file_size_bytes", ""),
                d.get("recorded_at", ""),
                "yes" if (d.get("recorded_at") and not is_midnight(d["recorded_at"])) else "no",
                len(tags_by_media[d["id"]]),
                len(progress_by_media[d["id"]]),
                d["created_at"],
                "",
            ])

print(f"Exported cleanup plan to: {output_path}")
print(f"  {len(cleanup_plan)} groups, {sum(len(p['dupes']) for p in cleanup_plan)} records marked DELETE")

Exported cleanup plan to: C:\Users\minds\Videos\DANCE\dance-library\TESTING\dedup_plan.csv
  119 groups, 119 records marked DELETE


---
## Step 5: Execute Cleanup

For each duplicate group:
1. **Transfer** metadata from dupes → keeper (recorded_at, tags, watch progress, etc.)
2. **Delete** duplicate DB records (cascade deletes media_tags and watch_progress for the dupe)
3. **Delete** duplicate R2 files (video + thumbnail)

**Run only after reviewing the plan above.**

In [9]:
# ── Execute cleanup ──

transferred = 0
deleted_db = 0
deleted_r2 = 0
errors = []

for plan in cleanup_plan:
    keeper = plan["keeper"]
    keeper_id = keeper["id"]

    # ── 1. Transfer metadata to keeper ──
    for t in plan["transfers"]:
        try:
            if t["type"] == "recorded_at":
                sb.table("media").update({"recorded_at": t["value"]}).eq("id", keeper_id).execute()
                transferred += 1

            elif t["type"] == "description":
                sb.table("media").update({"description": t["value"]}).eq("id", keeper_id).execute()
                transferred += 1

            elif t["type"] == "duration":
                sb.table("media").update({"duration": t["value"]}).eq("id", keeper_id).execute()
                transferred += 1

            elif t["type"] == "resolution":
                sb.table("media").update({"resolution": t["value"]}).eq("id", keeper_id).execute()
                transferred += 1

            elif t["type"] == "tags":
                for tag in t["details"]:
                    sb.table("media_tags").insert({
                        "media_id": keeper_id,
                        "tag_id": tag["tag_id"],
                        "start_time": tag.get("start_time"),
                        "end_time": tag.get("end_time"),
                    }).execute()
                transferred += t["count"]

            elif t["type"] == "watch_progress":
                for p in t["details"]:
                    sb.table("watch_progress").upsert({
                        "user_id": p["user_id"],
                        "media_id": keeper_id,
                        "position": p["position"],
                    }).execute()
                transferred += t["count"]

        except Exception as e:
            errors.append(("transfer", keeper_id, t["type"], str(e)))

    # ── 2. Delete duplicate DB records + R2 files ──
    for d in plan["dupes"]:
        dupe_id = d["id"]
        storage_path = d.get("storage_path", "")
        thumb_path = d.get("thumbnail_path", "")

        try:
            # Delete DB record (cascades to media_tags + watch_progress)
            sb.table("media").delete().eq("id", dupe_id).execute()
            deleted_db += 1

            # Delete video from R2
            if storage_path:
                try:
                    r2.delete_object(Bucket=R2_BUCKET_NAME, Key=storage_path)
                    deleted_r2 += 1
                except Exception:
                    pass

            # Delete thumbnail from R2
            if thumb_path:
                try:
                    r2.delete_object(Bucket=R2_BUCKET_NAME, Key=thumb_path)
                    deleted_r2 += 1
                except Exception:
                    pass

        except Exception as e:
            errors.append(("delete", dupe_id, d.get("original_filename"), str(e)))

print(f"Metadata transferred: {transferred}")
print(f"DB records deleted:   {deleted_db}")
print(f"R2 objects deleted:   {deleted_r2}")
if errors:
    print(f"\nErrors ({len(errors)}):")
    for err in errors:
        print(f"  {err}")
else:
    print("\nNo errors.")

Metadata transferred: 1
DB records deleted:   119
R2 objects deleted:   238

No errors.


In [10]:
# ── Storage savings estimate ──

total_dupe_bytes = 0
for plan in cleanup_plan:
    for d in plan["dupes"]:
        total_dupe_bytes += d.get("file_size_bytes", 0) or 0

print(f"Estimated R2 storage savings: {format_bytes(total_dupe_bytes)}")
print(f"(from removing {sum(len(p['dupes']) for p in cleanup_plan)} duplicate videos + their thumbnails)")

Estimated R2 storage savings: 15.2 GB
(from removing 119 duplicate videos + their thumbnails)
